# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Research question:** Which content pages should a constrained SEO/content team review first for refresh or related intervention, using observable search, freshness, engagement, and content signals while keeping the recommendation explainable and leakage-aware?

**Decision supported:** order a limited review queue. The output is not an automatic edit instruction and does not claim that refreshing a flagged page causes recovery.

In [1]:
import os, json, subprocess
from pathlib import Path
import pandas as pd

REPO_URL="https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR="flyrank-ml-internship"

def find_repo_root():
    here=Path.cwd().resolve()
    for candidate in [here,*here.parents]:
        if (candidate/"data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    return None

root=find_repo_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],check=True)
    root=Path(REPO_DIR).resolve()
os.chdir(root)

df=pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Research question ready.")
print("Starter rows:",len(df),"Clients:",df["client_id"].nunique())


Research question ready.
Starter rows: 30000 Clients: 32


## 2. Data

This submission's **executed modeling evidence** uses the repository's bundled public-safe starter slice:

- **30,000** pseudonymized content items
- **32** pseudonymized clients
- **44** fields
- trailing-90-day search/engagement context
- no raw client names, domains, URLs, keywords, queries, or titles

The broader gated FlyRank internship warehouse is documented at **78,835,655 daily fact rows**, plus content/client dimensions, spanning **2025-01-27 through 2026-06-30**. I do not claim to have run the gated warehouse scan in this submission unless an `HF_TOKEN` query receipt exists.

The starter target is a same-window proxy: `trend_direction == "down"`. It is useful for demonstrating the workflow but weaker than a properly separated future outcome.

In [2]:
data_receipt={
    "starter_rows":int(len(df)),
    "starter_clients":int(df["client_id"].nunique()),
    "starter_columns":int(df.shape[1]),
    "raw_identifying_fields_published":False,
    "warehouse_daily_fact_documented_rows":78835655,
    "warehouse_date_min":"2025-01-27",
    "warehouse_date_max":"2026-06-30",
    "warehouse_query_claimed_executed":False
}
print(json.dumps(data_receipt,indent=2))


{
  "starter_rows": 30000,
  "starter_clients": 32,
  "starter_columns": 44,
  "raw_identifying_fields_published": false,
  "warehouse_daily_fact_documented_rows": 78835655,
  "warehouse_date_min": "2025-01-27",
  "warehouse_date_max": "2026-06-30",
  "warehouse_query_claimed_executed": false
}


## 3. Methodology

The project follows a decision-first pipeline:

1. **Task:** ranking/scoring — which pages should be reviewed first?
2. **Proxy target:** current starter decline proxy, explicitly separated from future causal claims.
3. **Frozen baseline:** transparent score from visibility, freshness risk, position opportunity, and depth gap.
4. **Model:** logistic regression and random forest; probabilities are used as ranking scores.
5. **Validation:** whole-client holdout with seed 42; train and test clients never overlap.
6. **Primary metric:** Precision@50, fixed before model comparison.
7. **Leakage audit:** no `trend_direction`, `trend_pct`, product decision flags, IDs, or future windows as model features.
8. **Action layer:** model orders the queue; feature-only reason codes and human review determine what happens next.

The same split and metric are used for the rule and model.

In [3]:
with open("work/outputs/w05_model_metrics.json","r",encoding="utf-8") as fh:
    w5=json.load(fh)
with open("work/outputs/w06_validation_receipt.json","r",encoding="utf-8") as fh:
    w6=json.load(fh)
with open("work/outputs/w07_playbook_metrics.json","r",encoding="utf-8") as fh:
    w7=json.load(fh)

print("Split:",w5["split"])
print("Features:",len(w5["features"]))
print("Forbidden features:",w6["leakage_audit"]["forbidden_features_present"])
print("Human review required:",w7["human_review_required"])


Split: client_holdout
Features: 12
Forbidden features: []
Human review required: True


## 4. Results (vs baseline)

All headline ranking results below come from the **same client-holdout split**.

The transparent rule is the minimum benchmark. Logistic Regression and Random Forest are compared at the operational top of the queue using Precision@20/50/100. I report the measured result from this environment rather than borrowing the starter repository's reference number from a different library version.

In [4]:
results=pd.DataFrame(w5["results"])
print(results.round(3).sort_values("p50",ascending=False).to_string(index=False))
print("\nHeld-out test base rate:",round(w5["test_base_rate"],3))
print("Best by Precision@50:",w5["best_by_p50"])
print("Action queue top-50 proxy precision after full-data ranking:",round(w7["top50_proxy_precision"],3))


             method  p20  p50  p100  avg_precision  roc_auc
      random_forest 0.90 0.86  0.84          0.666    0.765
logistic_regression 0.90 0.78  0.78          0.632    0.716
         fixed_rule 0.15 0.26  0.35          0.469    0.628

Held-out test base rate: 0.391
Best by Precision@50: random_forest
Action queue top-50 proxy precision after full-data ranking: 1.0


## 5. Limitations

This work **cannot** claim:

- that any feature is a Google ranking factor,
- that a refresh causes recovery,
- that the same-window starter proxy is equivalent to a future production target,
- that the 30,000-row starter slice represents every client or period,
- or that a high score should trigger an automatic edit.

The strongest next research step is a past→future warehouse label with non-overlapping windows, time-forward evaluation, and a sealed final month. After that, the causal value of acting on the queue would still require a controlled or otherwise credible intervention design.

The current result is best read as: **a measured decision-support ranking experiment on a public-safe starter slice, with grouped validation and explicit leakage controls.**

In [5]:
limits=[
    "no causal refresh claim",
    "same-window proxy target",
    "starter-slice population",
    "no automatic action",
    "warehouse time-forward validation still required"
]
print("Limitations documented:",len(limits))
for x in limits: print("-",x)


Limitations documented: 5
- no causal refresh claim
- same-window proxy target
- starter-slice population
- no automatic action
- warehouse time-forward validation still required


## 6. Ranked recommendations

The action playbook recommends a **review sequence**, not edits:

1. Start with high-ranked candidates where several transparent reason codes agree.
2. For low CTR with real visibility, review title/meta/intent and SERP context.
3. For stale visible pages, review whether the content is still accurate and strategically relevant.
4. For weak engagement, verify intent match and measurement coverage before changing content.
5. For thin visible pages, review topical completeness — do not lengthen content mechanically.
6. Route uncertain single-signal cases to monitoring.
7. Never auto-rewrite, delete, redirect, merge, or publish from the score.

Each recommendation remains subject to seasonality, consolidation, tracking quality, and editorial context.

In [6]:
print("Rows ranked:",w7["rows_ranked"])
print("Top-50 proxy precision:",round(w7["top50_proxy_precision"],3))
print("\nAction mix:")
for action,count in w7["action_counts"].items():
    print(f"- {action}: {count:,}")
print("\nAutomatic editing:",w7["automatic_editing"])


Rows ranked: 30000
Top-50 proxy precision: 1.0

Action mix:
- monitor_only: 12,406
- review_title_meta_and_intent: 9,759
- review_for_refresh: 4,356
- review_engagement_and_intent: 3,415
- review_depth_and_coverage: 64

Automatic editing: False


## 7. Artifacts the paper embeds

The public paper reuses:
- the model-vs-baseline comparison table,
- the action-mix SVG from Week 7,
- the validation/leakage receipt,
- and the explicit limitations and human-review policy.

The paper is generated as a responsive static page under `docs/index.html`. The page contains all nine required research-paper sections: **Title + Abstract, Introduction, Data, Methodology, Results, Limitations, Ranked Recommendations, Reproducibility, and Acknowledgments & Data Credit.**

In [7]:
from html import escape

rule=next(r for r in w5["results"] if r["method"]=="fixed_rule")
logit=next(r for r in w5["results"] if r["method"]=="logistic_regression")
rf=next(r for r in w5["results"] if r["method"]=="random_forest")
best=max(w5["results"],key=lambda r:r["p50"])

paper_title="Which Content Pages Should Be Reviewed First? An Explainable Refresh-Opportunity Ranking Study"

html=f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{escape(paper_title)}</title>
<style>
:root{{--bg:#0b1220;--card:#111b2e;--text:#eaf0f8;--muted:#aab7ca;--accent:#59d8c5;--line:#26344d;}}
*{{box-sizing:border-box}} body{{margin:0;background:var(--bg);color:var(--text);font-family:Inter,system-ui,Arial,sans-serif;line-height:1.65}}
main{{max-width:980px;margin:auto;padding:48px 24px 80px}} h1{{font-size:clamp(2rem,5vw,3.6rem);line-height:1.08;margin:.2em 0}} h2{{margin-top:2.2em;color:var(--accent)}}
p,li{{color:#d9e2ef}} .muted{{color:var(--muted)}} .hero{{padding-bottom:28px;border-bottom:1px solid var(--line)}}
.badge{{display:inline-block;padding:6px 10px;border:1px solid var(--line);border-radius:999px;color:var(--accent);font-size:.84rem}}
.abstract,.callout{{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:20px 22px;margin:20px 0}}
.grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:12px}} .metric{{background:var(--card);border:1px solid var(--line);padding:16px;border-radius:12px}}
.metric b{{font-size:1.8rem;display:block;color:var(--accent)}} table{{width:100%;border-collapse:collapse;margin:18px 0;background:var(--card)}}
th,td{{padding:11px 12px;border-bottom:1px solid var(--line);text-align:left}} th{{color:var(--accent)}} code{{background:#18243b;padding:2px 5px;border-radius:5px}}
a{{color:var(--accent)}} img{{max-width:100%}} .figure{{background:white;border-radius:12px;padding:12px;margin:18px 0}}
footer{{margin-top:50px;padding-top:22px;border-top:1px solid var(--line);color:var(--muted)}}
@media(max-width:700px){{main{{padding:28px 16px 60px}}.grid{{grid-template-columns:1fr}}table{{display:block;overflow:auto}}}}
</style>
</head>
<body><main>
<header class="hero"><span class="badge">FlyRank ML Internship · Applied ML Research</span>
<h1>{escape(paper_title)}</h1>
<p class="muted">A public-safe capstone by Abdallah · reproducible from the linked repository · decision-support, not causal automation.</p></header>

<section class="abstract"><h2>Abstract</h2>
<p>Which pages should an SEO/content team review first when review capacity is limited? I frame the problem as ranking/scoring and evaluate a transparent rule against learned models on a 30,000-row anonymized starter slice spanning 32 pseudonymized clients. The target in this executed study is a same-window decline proxy, so the result is intentionally scoped as a workflow and ranking experiment rather than a future-outcome or causal study. Validation holds out entire clients, the primary metric is Precision@50, and label-derived fields are excluded from the feature set. On this run the best method by Precision@50 is <b>{escape(best["method"])}</b> at <b>{best["p50"]:.3f}</b>, compared with the fixed rule at <b>{rule["p50"]:.3f}</b>; the output is converted into a reason-coded human review queue with explicit no-go automation rules.</p></section>

<section><h2>1. Introduction / Problem</h2>
<p>Large content portfolios create a prioritization problem before they create a modeling problem. A strategist may have thousands of pages but capacity to inspect only a small set. The useful decision is therefore not “predict decline” in isolation; it is <b>which pages should be reviewed first</b>, with enough explanation that a human can decide what, if anything, to change.</p>
<p>This capstone follows the lane selected in Week 1: Refresh / Content Opportunity Scoring. It preserves the original decision, actor, action, and error-cost framing throughout the later modeling work.</p></section>

<section><h2>2. Data</h2>
<div class="grid"><div class="metric"><b>30,000</b>content items</div><div class="metric"><b>32</b>pseudonymized clients</div><div class="metric"><b>44</b>starter fields</div></div>
<p>The executed evidence uses the bundled anonymized starter slice with trailing-90-day search, engagement, freshness, and content measurements. No real client names, domains, URLs, keywords, queries, or titles are published. The larger gated internship warehouse is documented separately at 78,835,655 daily fact rows from 2025-01-27 through 2026-06-30; this page does not claim a gated full-warehouse scan without an authentication-backed query receipt.</p></section>

<section><h2>3. Methodology</h2>
<ul>
<li><b>Task:</b> ranking/scoring for a top-K review queue.</li>
<li><b>Starter proxy:</b> <code>trend_direction == "down"</code>; its source columns are never model features.</li>
<li><b>Baseline:</b> frozen rule combining visibility, freshness risk, position opportunity, and depth gap.</li>
<li><b>Models:</b> Logistic Regression and Random Forest over 12 observable feature-time fields.</li>
<li><b>Validation:</b> whole-client holdout, seed 42, zero client overlap.</li>
<li><b>Metric:</b> Precision@50 fixed before comparison, with Precision@20/100 and base rate shown alongside it.</li>
<li><b>Leakage controls:</b> no label siblings, IDs-as-features, product decision flags, or future-window inputs.</li>
</ul></section>

<section><h2>4. Results</h2>
<table><thead><tr><th>Method</th><th>P@20</th><th>P@50</th><th>P@100</th><th>Avg Precision</th><th>ROC AUC</th></tr></thead><tbody>
<tr><td>Fixed rule</td><td>{rule["p20"]:.3f}</td><td>{rule["p50"]:.3f}</td><td>{rule["p100"]:.3f}</td><td>{rule["avg_precision"]:.3f}</td><td>{rule["roc_auc"]:.3f}</td></tr>
<tr><td>Logistic Regression</td><td>{logit["p20"]:.3f}</td><td>{logit["p50"]:.3f}</td><td>{logit["p100"]:.3f}</td><td>{logit["avg_precision"]:.3f}</td><td>{logit["roc_auc"]:.3f}</td></tr>
<tr><td>Random Forest</td><td>{rf["p20"]:.3f}</td><td>{rf["p50"]:.3f}</td><td>{rf["p100"]:.3f}</td><td>{rf["avg_precision"]:.3f}</td><td>{rf["roc_auc"]:.3f}</td></tr>
</tbody></table>
<p>Held-out client test base rate: <b>{w5["test_base_rate"]:.3f}</b>. The result should be read as measured ranking performance on this proxy and population, not as evidence that an edit will cause recovery.</p>
<div class="figure"><img src="w07_action_mix.svg" alt="Action playbook mix"></div>
<p class="muted">Action mix from the full-data review queue. Model = ordering; reason codes + human context = action review.</p></section>

<section><h2>5. Limitations & Honest Framing</h2>
<ul><li>The starter target is same-window, not a future observed outcome.</li><li>The 30k slice is not the full 78.8M-row daily warehouse.</li><li>Client grouping helps generalization testing but does not replace time-forward evaluation.</li><li>Seasonality, consolidation, SERP changes, campaigns, and tracking artifacts can make a recommendation wrong.</li><li>No causal claim is made about refreshing, rewriting, or otherwise changing a page.</li></ul>
<div class="callout"><b>Safe claim:</b> the system measured how well alternative ranking methods prioritize a defined starter proxy on held-out clients. It supports human review ordering only.</div></section>

<section><h2>6. Ranked Recommendations</h2>
<ol><li>Review high-ranked multi-signal candidates first.</li><li>For visible low-CTR pages, inspect snippet/intent/SERP context before changing metadata.</li><li>For stale visible pages, inspect accuracy and strategic relevance before refreshing.</li><li>For weak-engagement pages, confirm analytics coverage and intent match first.</li><li>For thin visible pages, inspect topical completeness rather than adding words mechanically.</li><li>Route uncertain single-signal cases to monitoring.</li></ol>
<p><b>No-go:</b> no automatic rewrites, publishing, deletion, redirects, merges, or client actions from the score.</p></section>

<section><h2>7. Reproducibility</h2>
<p>All notebooks live under <code>work/notebooks/</code>. Random seeds are fixed at 42. Small JSON metric receipts and paper figures are committed; bulk queue CSVs remain git-ignored by design. The repository's smoke test checks the pipeline and leak guard.</p>
<p><a href="https://github.com/3bud-ZC/flyrank-ml-internship">View the full repository and executed notebooks</a></p></section>

<section><h2>8. Acknowledgments & Data Credit</h2>
<p>Built as part of the FlyRank Machine Learning Internship using the pseudonymized FlyRank internship starter data and release documentation. Data/product credit: <a href="https://flyrank.ai">FlyRank AI</a>. The public research report reviewed during the claim-audit week is <a href="https://state-of-seo-2026.flyrank.ai/">The State of AI-Driven SEO in Numbers</a>.</p></section>

<footer>Last generated from committed metric receipts. Claims are observational / predictive decision-support unless a stronger design is explicitly named.</footer>
</main></body></html>"""

Path("docs").mkdir(exist_ok=True)
figure_src = Path("work/figures/w07_action_mix.svg")
figure_dst = Path("docs/w07_action_mix.svg")
figure_dst.write_text(figure_src.read_text(encoding="utf-8"), encoding="utf-8")
Path("docs/index.html").write_text(html,encoding="utf-8")
print("Generated docs/index.html")
print("Paper sections present:",9)


Generated docs/index.html
Paper sections present: 9


## Self-check

- [x] Every capstone section is filled with markdown thinking and executable evidence
- [x] No client names, URLs, private queries, or credentials are published
- [x] Claims use observed / measured / decision-support language
- [x] Paper generator contains all 9 required sections
- [ ] Notebook runs top to bottom with no errors
- [ ] Deployed paper URL recorded in `submission/paper_url.txt`
- [ ] Demo outline + two shareable cuts committed under `work/`

---

## ML-12 — Tell the Story

### 5-minute demo outline
**0:00–0:40 — Question.** Which content pages should a limited SEO/content team review first?

**0:40–1:30 — Data and boundaries.** 30,000 anonymized content items across 32 pseudonymized clients; same-window decline proxy; no raw client/URL/query data.

**1:30–2:30 — Method.** Freeze a transparent baseline, hold out whole clients, compare learned ranking on Precision@50, audit leakage.

**2:30–3:25 — One result.** Show the model-vs-rule table and one action-mix chart. State the exact measured Precision@50 and test base rate.

**3:25–4:15 — Honest interpretation.** The ranking improves review prioritization for this proxy/population; it does not show that refreshing causes recovery.

**4:15–5:00 — Recommendation.** Use model scores to order review; keep feature-only reason codes and mandatory human checks; next step is a non-overlapping past→future warehouse target.

### Social-post cut
I built an explainable content-review ranking workflow on 30,000 anonymized content items. Instead of optimizing for generic accuracy, I framed the real decision as “which pages should an SEO team inspect first?” and evaluated a transparent rule against learned ranking using client-holdout Precision@50. The key lesson was not just the score: validation boundaries, leakage checks, reason codes, and human review determine whether an ML output is actually usable. Full write-up: **https://flyrank-ml-paper-production.up.railway.app/**

### Employer-facing 3-sentence summary
I built an end-to-end ML decision-support pipeline that ranks content pages for review using 30,000 anonymized production-shaped records across 32 pseudonymized clients. I designed a transparent baseline, trained and compared models under client-grouped validation, audited leakage, and converted the ranking into a reason-coded human action playbook with reproducible metric receipts. The result is deployed as a public research paper with explicit limitations and a clear path toward a time-forward warehouse evaluation.

In [8]:
demo="""# ML-12 — 5-Minute Demo Outline

1. **Question (0:00–0:40):** Which content pages should a limited SEO/content team review first?
2. **Data (0:40–1:30):** 30,000 anonymized content items, 32 pseudonymized clients; same-window decline proxy; public-safe fields only.
3. **Method (1:30–2:30):** frozen transparent baseline → client-holdout split → Logistic Regression / Random Forest → Precision@50 → leakage audit.
4. **Result (2:30–3:25):** show the committed model-vs-baseline table and the action-mix chart. Quote only the measured values from the current run.
5. **Honest interpretation (3:25–4:15):** decision-support ranking, not causal proof that refresh causes recovery.
6. **Recommendation (4:15–5:00):** use model = order, reason codes = explanation, human = decision. Next: non-overlapping warehouse target + time-forward validation.
"""
social="""# Social Post Cut

I built an explainable content-review ranking workflow on 30,000 anonymized content items.

Instead of optimizing for generic accuracy, I framed the real decision as: **which pages should an SEO team inspect first?** I froze a transparent baseline, compared learned ranking under client-holdout validation using Precision@50, audited leakage, and turned the result into a reason-coded human review queue.

The biggest lesson: the model score is only one part of a useful ML system. Validation boundaries, reason codes, limits, and a clear no-auto-edit policy matter just as much.

Full research paper: https://flyrank-ml-paper-production.up.railway.app/
"""
employer="""# Employer-Facing Summary

I built an end-to-end ML decision-support pipeline that ranks content pages for review using 30,000 anonymized production-shaped records across 32 pseudonymized clients.

I designed a transparent baseline, trained and compared models under client-grouped validation, audited leakage, and converted the ranking into a reason-coded human action playbook with reproducible metric receipts.

The result is deployed as a public research paper with explicit limitations and a clear next step toward a non-overlapping, time-forward warehouse evaluation.
"""

Path("work/demo_outline.md").write_text(demo)
Path("work/social_post.md").write_text(social)
Path("work/employer_summary.md").write_text(employer)
print("Created ML-12 shareable cuts under work/.")


Created ML-12 shareable cuts under work/.
